# Test récupération logos Volleybox
Ce notebook valide la méthode de récupération via le scraper Volleybox (matching + extraction logo).

In [7]:
from pyvolley.scrapers.volleybox import VolleyboxLogoScraper
from pyvolley.database.connection import DatabaseSession, init_db
from pyvolley.database.models import ClubDB
from sqlalchemy import select

In [8]:
init_db()
scraper = VolleyboxLogoScraper()
print('scraper prêt')

scraper prêt


In [9]:
with DatabaseSession() as session:
    clubs = session.execute(
        select(ClubDB).where(ClubDB.code_ffvb.is_not(None)).order_by(ClubDB.nom.asc()).limit(20)
    ).scalars().all()

results = []
for club in clubs:
    names = [club.nom]
    if club.nom_court:
        names.append(club.nom_court)
    candidate = scraper.find_logo_for_club(names)
    results.append((club.nom, candidate.team_url if candidate else None, candidate.logo_url if candidate else None, candidate.score if candidate else None))

results[:10]

[('A.S. DE L&#039', None, None, None),
 ('A.S. VALLEE DE LA SAUER', None, None, None),
 ('A.S.L. VENDEE VOLLEY-BALL', None, None, None),
 ('ACBB', None, None, None),
 ('AGDE VOLLEY BALL', None, None, None),
 ('AGGLO SUD VOLLEY-BALL 76', None, None, None),
 ('AIX UNIVERSITE CLUB 13 VOLLEY-BALL', None, None, None),
 ('AL CAUDRY VOLLEY-BALL', None, None, None),
 ('ALERTE SPORTIVE MONTLOUIS', None, None, None),
 ('ALTERNA STADE POITEVIN VOLLEY BALL', None, None, None)]

In [10]:
# Appliquer sur un petit batch (décommenter pour persister)
# with DatabaseSession() as session:
#     updated = 0
#     for club in clubs:
#         names = [club.nom, club.nom_court] if club.nom_court else [club.nom]
#         candidate = scraper.find_logo_for_club(names)
#         if candidate and candidate.score >= 0.35 and candidate.logo_url:
#             club.logo_url = candidate.logo_url
#             updated += 1
#     session.commit()
# print(f'logos mis à jour: {updated}')

In [11]:
# Vérification ciblée d'un club connu
candidate = scraper.find_logo_for_club(["HARNES VOLLEY-BALL", "HARNES VB"])
(
    candidate.team_url if candidate else None,
    candidate.logo_url if candidate else None,
    candidate.score if candidate else None,
)

(None, None, None)

In [12]:
import importlib
import pyvolley.scrapers.volleybox.logo_scraper as vb_logo_scraper

importlib.reload(vb_logo_scraper)
scraper = vb_logo_scraper.VolleyboxLogoScraper()